In [ ]:
import os
import shutil
import tempfile
import zipfile
import pathlib as Path
from concurrent.futures import ThreadPoolExecutor

import scanpy as sc
import hisepy as hp
from spatialdata_io import xenium

import lamindb as ln

ln.track("3W10mqLS5IEx")

In [ ]:
SAMPLES = {
    'TIS04399-001': ('e841cbfc-851f-4781-a707-5cf06e3e6068', 'FH1001', 'MM Pre-Treatment'),
    'TIS02434-001': ('f2957fe1-aa58-4d75-adfe-1cabd4f9bbe0', 'FH1009', 'MM Pre-Treatment'),
    'TIS02437-001': ('cba78e0f-62d1-4592-9581-a02a5cea0fae', 'FH1011', 'MM Pre-Treatment'),
    'TIS04398-001': ('b8e8fe26-5d7c-4aee-9850-2651a7073ceb', 'FH1015', 'MM Pre-Treatment'),
    'TIS02611-001': ('8460b141-a598-47c1-a79e-d30658698302', 'FH1016', 'MM Pre-Treatment'),
    'TIS02603-001': ('d14fff16-f2e1-4dcd-bf11-fcbfe364dada', 'FH1017', 'MM Pre-Treatment'),
    'TIS02604-001': ('fee36c64-5a50-4b7f-868e-616088d9701f', 'FH1018', 'MM Pre-Treatment'),
    'TIS02613-001': ('69ea81f2-1751-43ad-baf0-1952f593f4e6', 'FH1021', 'MM Pre-Treatment'),
    'TIS02421-001': ('91211de2-f801-4f2b-af3a-18017dc44005', 'FH1003', 'MM Pre-Treatment'),
    'TIS02430-001': ('c3e68735-26be-471c-834e-24833fbde0dd', 'FH1007', 'MM Pre-Treatment'),
    'TIS02424-001': ('40a0ef40-6b5a-43db-845e-d5bc58faa685', 'FH1004', 'MM Pre-Treatment'),
    'TIS02439-001': ('807357ef-5733-427f-b6ce-b828754f180c', 'FH1012', 'MM Pre-Treatment'),
    'TIS02594-001': ('f3c5af40-e58c-4143-86ce-0194ca9eab26', 'FH1006', 'MM Pre-Treatment'),
    # TIS04397-001 (FH1010) dropped: it is on a 377-gene panel, not the 480-gene
    # panel of the other 13, which would collapse the join='inner' gene set to 258.
}

In [ ]:
ingest_label      = ln.ULabel(name='ingest').save()
raw_label         = ln.ULabel(name='raw').save()
analysis_label    = ln.ULabel(name='fh1_pretreatment').save()

In [ ]:
OUTS_DIR = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/outs'

def fetch_sample(name):
    sample_dir = os.path.join(OUTS_DIR, name)
    if os.path.isdir(sample_dir):
        return
    (zip_path, ) = hp.cache_files(file_ids=[SAMPLES[name][0]])
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        extracted_folder = next(
            os.path.join(tmp, d) for d in os.listdir(tmp)
            if os.path.isdir(os.path.join(tmp, d))
        )
        shutil.move(extracted_folder, sample_dir)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch_sample, SAMPLES))

In [ ]:
adatas = {}
for name, (uuid, subject, timepoint) in SAMPLES.items():
    sample_dir = os.path.join(OUTS_DIR, name)
    print(sample_dir)

    adata = xenium(
        sample_dir,
        morphology_focus=False,
        morphology_mip=False,
        cells_boundaries=False,
        nucleus_boundaries=False,
        cells_labels=False,
        nucleus_labels=False,
        transcripts=False,
    ).tables['table']
    adata.obs_names_make_unique()
    adata.obs['sample'] = name
    adata.obs['cohort_subject'] = subject
    adata.obs['treatment_timepoint'] = timepoint
    
    artifact = ln.Artifact.from_anndata(adata, key=f'{name}.h5ad', description='raw unprocessed h5ads from fh1_pretreatment outs. TIS04397-001 (FH1010) dropped: it is on a 377-gene panel, not the 480-gene').save()
    artifact.ulabels.add(ingest_label, raw_label, analysis_label)
    adatas[name] = adata

In [ ]:
combined = sc.concat(adatas, label='sample', join='inner')
combined.obs_names_make_unique()

artifact = ln.Artifact.from_anndata(
    combined,
    key='combined_raw.h5ad',
    description='all raw unprocessed h5ads from outs combined. no further processing'
).save()
artifact.ulabels.add(ingest_label, raw_label, analysis_label)

ln.finish()